In [2]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [19]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater, VaspBuilderUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm
from matchest.aiida_utils.pmg import load_mp_struct
from ase.symbols import Formula
from aiida.engine import submit

from matchest.aiida_utils.workflows.simple_vac import SimpleVacancyWorkChain

In [20]:
basepath = GroupPathX('hc-defect')
workpath = basepath['workflows']
elemental_struct_path = GroupPathX('defects/elemental_ref')

In [21]:
elemental_struct_path.show_tree()

elemental_ref
├── As_bulk *
├── Au_bulk *
├── Ba_bulk *
├── Bi_bulk *
├── Ca_bulk *
├── Cd_bulk *
├── Cu_bulk *
├── Ge_bulk *
├── Hg_bulk *
├── In_bulk *
├── Mg_bulk *
├── O_bulk *
├── P_bulk *
├── Pb_bulk *
├── S_bulk *
├── Sb_bulk *
├── Se_bulk *
├── Sn_bulk *
├── Sr_bulk *
├── Te_bulk *
├── Tl_bulk *
└── Zn_bulk *



## Define the input to the wokchain

Generate structure

In [22]:
# For antipervoskite
form = Formula('Sr3SnO')  # Change the formula here
symbols = list(form)
ASite = symbols[0]
BSite = symbols[3]
CSite = symbols[4]

structure = load_mp_struct('mp-19944')

ps = structure.get_pymatgen()
ps['Sr'] = 'He'
ps['Pb'] = 'Ne' 
ps['O'] = 'Xe' 

ps['He'] = ASite  
ps['Ne'] = BSite 
ps['Xe'] = CSite 

structure = orm.StructureData(pymatgen=ps)
print(ps)

Full Formula (Sr3 Sn1 O1)
Reduced Formula: Sr3SnO
abc   :   5.172510   5.172510   5.172510
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (5)
  #  SP      a    b    c
---  ----  ---  ---  ---
  0  Sr    0.5  0    0.5
  1  Sr    0    0.5  0.5
  2  Sr    0.5  0.5  0
  3  Sn    0    0    0
  4  O     0.5  0.5  0.5


In [23]:
builder = SimpleVacancyWorkChain.get_builder()

upd = VaspRelaxUpdater(builder = builder.relax, ).apply_preset(structure, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1, 'ncore':8 , 'kpar': 4}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600*12, queue_name='xhhctdnormal')
upd.set_label(f'{form} RELAX')
upd.set_relax_settings(algo='rd')
# Assign the elemental structures
builder.elemental_structures = {key: elemental_struct_path[key + '_bulk'].node  for
                                key in set(symbols)}
assert None not in builder.elemental_structures.values()
builder.supercell_dim = orm.List([2,2,2])  # 222 supercell
# Updated parameters for supercell calculation
builder.supercell_workchain_updates = orm.Dict(
    {'options': {
        'resources': {'tot_num_mpiprocs': 128, 'num_machines':2},
        'max_wallclock_seconds': 3600 * 12, 
        'queue_name': 'xhhctdnormal',
        },
     'incar': {'kpar': 2, 'ncore': 8, 'isym': 0}
    }
)

In [24]:
builder

Process class: SimpleVacancyWorkChain
Inputs:
elemental_structures:
  O: O
  Sn: Sn
  Sr: Sr
metadata: {}
relax:
  metadata:
    label: Sr3SnO RELAX
  relax_settings:
    algo: rd
    clean_reuse: true
    convergence_absolute: false
    convergence_max_iterations: 5
    convergence_mode: last
    convergence_on: true
    convergence_positions: 0.1
    convergence_shape_angles: 0.1
    convergence_shape_lengths: 0.1
    convergence_volume: 0.01
    double_relax_mode: false
    energy_cutoff: null
    force_cutoff: 0.03
    hybrid_calc_bootstrap: false
    hybrid_calc_bootstrap_wallclock: 3600
    keep_magnetization: false
    keep_sp_workdir: false
    perform: true
    positions: true
    reuse: false
    shape: true
    steps: 60
    volume: true
  structure: OSnSr3
  vasp:
    code: vasp-6.3.2@sugon-xh-v2
    dynamics: {}
    kpoints_spacing: 0.05
    metadata: {}
    options:
      max_wallclock_seconds: 43200
      queue_name: xhhctdnormal
      resources:
        num_machines: 1


In [25]:
running = submit(builder)

#workpath[f'{form}_work'] = running


In [26]:
workpath.show_tree(decorate_by=['exit_status', 'pk'])

workflows 121
├── Ba3PbO_work [0] | 668617
├── Ba3SnO_work [0] | 668797
├── Ca2Pb_work [0] | 669135
├── Ca2Sn_work [0] | 669224
├── Ca3Bi2_work [0] | 669702
├── Ca3PbO_work [0] | 668910
├── GeCdAs2_work [0] | 664271
├── GeCdSb2_work [0] | 664179
├── GeZnBi2_work [0] | 664223
├── GeZnSb2_work [excepted] | 664317
├── GeZnSb2_work_sym_update [0] | 665530
├── InAuSe2_work_sym_update [0] | 666598
├── InCuSe2_work_sym_update [0] | 670540
├── Mg2Pb_work [501] | 670744
├── Mg2Sn_work [501] | 670766
├── Mg3Bi2_work [0] | 669492
├── Mg3PbO_work [0] | 668955
├── PbCdAs2_work [excepted] | 664341
├── PbCdAs2_work_sym_update [0] | 665552
├── PbCdP2_work [excepted] | 664132
├── PbCdP2_work_sym_update [0] | 665574
├── PbZnAs2_work [excepted] | 664157
├── PbZnAs2_work_sym_update [0] | 665596
├── PbZnP2_work [excepted] | 664249
├── PbZnP2_work_sym_update [0] | 665618
├── PbZnSb2_work [0] | 663926
├── SnCdSb2_work [0] | 664110
├── SnZnBi2_work [0] | 664201
├── SnZnSb2_work [excepted] | 664293
├── SnZnSb2

In [27]:
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pymatgen.core import Composition

In [28]:
results = defaultdict(lambda : {})
for path in workpath:
    node = path.node
    if node.is_finished_ok:
        for kind, eng in node.outputs.vacancy_formation_energies.items():
            ref_structure = node.inputs.relax.structure.get_ase()
            form = ref_structure.symbols.formula.reduce()[0]
            #print(ref_structure.symbols.formula.reduce()[0], kind, eng[kind])
            results[str(form)][kind] = eng[kind]

Find the minimum as maxium V formation energy for cations

In [29]:
min_max = []
for key,value in results.items():
    
    symbols = list(set(Formula(key)))
    comp = Composition(key)
    sorted_elems = sorted(comp.elements, key=lambda x : x.electron_affinity)
    # Discard the element with the higest electron affinity
    no_anion = [x.symbol for x in sorted_elems[:-1]]
    #print(f"{key}  max: {max(engs):.3f} eV min: {min(engs):.3f} eV")
    #print(sorted_elems)
    names, engs = zip(*[[f'V_{a}' , value[f'V_{a}']] for a in no_anion])
    min_max.append([key, max(engs), min(engs), names[np.argmax(engs)], names[np.argmin(engs)]])
    
df = pd.DataFrame(min_max, columns=['name', 'E_form_max', 'E_form_min', 'E_form_max_element', 'E_form_min_element'])
df = df.set_index('name')
df

,E_form_max,E_form_min,E_form_max_element,E_form_min_element
name,,,,
PbZnSb2,1.770830,1.087918,V_Pb,V_Zn
SnCdSb2,1.322994,1.241505,V_Sb,V_Cd
GeCdSb2,1.427259,1.115496,V_Sb,V_Cd
SnZnBi2,1.276520,0.278016,V_Bi,V_Zn
GeZnBi2,1.456656,0.227689,V_Bi,V_Zn
GeCdAs2,1.837619,1.786082,V_Cd,V_As
GeZnSb2,1.586109,0.971533,V_Sb,V_Zn
PbCdAs2,1.711863,1.693158,V_Cd,V_Pb
PbCdP2,2.519697,0.884824,V_Cd,V_Pb


## Print the formation energies for easy copying into the summary feishu sheet

In [15]:
cases="""InAuSe2
TlCuSe2
PbZnSb2
PbZnSb2
GeCdSb2
SnCdSb2
PbCdP2
PbCdAs2
PbZnAs2
GeCdSb2
Mg3PbO
Ba3SnO
Ba3PbO
Mg3Bi2
Mg3Bi2
Sr2Pb
Ca2Pb"""
for value in df.loc[[x for x in cases.split('\n')]].E_form_min:
    print(f'{value:.3f}')

1.533
0.581
1.088
1.088
1.115
1.242
0.885
1.693
1.800
1.115
0.225
1.640
1.538
1.166
1.166
2.045
1.604


In [17]:
cases="""InCuSe2
InCuSe2
SnZnSb2
GeZnSb2
GeZnSb2
SnZnSb2
PbZnP2
GeCdAs2
PbZnP2
GeCdAs2
Ca3PbO
Sr3SnO
Sr3PbO
Ca3Bi2
Sr3Bi2
Ca2Pb
Ca2Sn"""
for value in df.loc[[x for x in cases.split('\n')]].E_form_min:
    print(f'{value:.3f}')

0.419
0.419
1.101
0.972
0.972
1.101
2.657
1.786
2.657
1.786
1.893
1.745
1.557
2.673
2.823
1.604
1.843
